In [ ]:
#!pip install meilisearch

In [9]:
import meilisearch
import json

client = meilisearch.Client('http://localhost:7700', MEILI_SEARCH_KEY)

In [6]:
import os
from dotenv import load_dotenv
load_dotenv()
MEILI_SEARCH_KEY = os.environ['MEILI_SEARCH_KEY']

In [5]:
import pandas as pd

In [11]:
df = pd.read_csv('./nasdaq_screener_1775713099848.csv', na_filter=False)
df.head(3)

,Symbol,Name,Last Sale,Net Change,% Change,Market Cap,Country,IPO Year,Volume,Sector,Industry
0,A,Agilent Technologies Inc. Common Stock,$116.92,3.04,2.669%,33041862904.00,United States,1999,1384565,Industrials,Biotechnology: Laboratory Analytical Instruments
1,AA,Alcoa Corporation Common Stock,$71.76,-1.20,-1.645%,18934772426.00,United States,2016,7131647,Industrials,Aluminum
2,AACB,Artius II Acquisition Inc. Class A Ordinary Sh...,$10.36,0.00,0.00%,0.00,United States,2025,106,,


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7082 entries, 0 to 7081
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Symbol      7082 non-null   str    
 1   Name        7082 non-null   str    
 2   Last Sale   7082 non-null   str    
 3   Net Change  7082 non-null   float64
 4   % Change    7082 non-null   str    
 5   Market Cap  7082 non-null   str    
 6   Country     7082 non-null   str    
 7   IPO Year    7082 non-null   str    
 8   Volume      7082 non-null   int64  
 9   Sector      7082 non-null   str    
 10  Industry    7082 non-null   str    
dtypes: float64(1), int64(1), str(9)
memory usage: 1.4 MB


In [15]:
# 전처리
df[df['Symbol'].str.contains(r'[^a-zA-X0-9-_/^ ]', regex=True)]
df['id'] = df['Symbol'].str.strip().replace(r'[/^]', '_', regex=True)
d = df.to_dict(orient='records')
d

[{'Symbol': 'A',
  'Name': 'Agilent Technologies Inc. Common Stock',
  'Last Sale': '$116.92',
  'Net Change': 3.04,
  '% Change': '2.669%',
  'Market Cap': '33041862904.00',
  'Country': 'United States',
  'IPO Year': '1999',
  'Volume': 1384565,
  'Sector': 'Industrials',
  'Industry': 'Biotechnology: Laboratory Analytical Instruments',
  'id': 'A'},
 {'Symbol': 'AA',
  'Name': 'Alcoa Corporation Common Stock ',
  'Last Sale': '$71.76',
  'Net Change': -1.2,
  '% Change': '-1.645%',
  'Market Cap': '18934772426.00',
  'Country': 'United States',
  'IPO Year': '2016',
  'Volume': 7131647,
  'Sector': 'Industrials',
  'Industry': 'Aluminum',
  'id': 'AA'},
 {'Symbol': 'AACB',
  'Name': 'Artius II Acquisition Inc. Class A Ordinary Shares',
  'Last Sale': '$10.36',
  'Net Change': 0.0,
  '% Change': '0.00%',
  'Market Cap': '0.00',
  'Country': 'United States',
  'IPO Year': '2025',
  'Volume': 106,
  'Sector': '',
  'Industry': '',
  'id': 'AACB'},
 {'Symbol': 'AACBR',
  'Name': 'Artius

# 마일리서치에 인덱스 만들고 추가

In [17]:
# 적재
client.index('nasdaq').add_documents(d, primary_key='id')

TaskInfo(task_uid=0, index_uid='nasdaq', status='enqueued', type='documentAdditionOrUpdate', enqueued_at=datetime.datetime(2026, 4, 9, 5, 48, 37, 754992))

# 인덱스에서 검색

In [ ]:
# client.index('nasdaq').search('PLTR')
client.index('nasdaq').search('PLT') # 유사한 종목

{'hits': [{'id': 'PLTK',
   'Symbol': 'PLTK',
   'Name': 'Playtika Holding Corp. Common Stock',
   'Last Sale': '$3.21',
   'Net Change': 0.07,
   '% Change': '2.229%',
   'Market Cap': '1217663122.00',
   'Country': 'Israel',
   'IPO Year': '2021',
   'Volume': 1224260,
   'Sector': 'Technology',
   'Industry': 'EDP Services'},
  {'id': 'PLTR',
   'Symbol': 'PLTR',
   'Name': 'Palantir Technologies Inc. Class A Common Stock',
   'Last Sale': '$140.76',
   'Net Change': -9.31,
   '% Change': '-6.204%',
   'Market Cap': '336510809280.00',
   'Country': 'United States',
   'IPO Year': '',
   'Volume': 64567570,
   'Sector': 'Technology',
   'Industry': 'Computer Software: Prepackaged Software'}],
 'query': 'PLT',
 'processingTimeMs': 2,
 'limit': 20,
 'offset': 0,
 'estimatedTotalHits': 2}